# Filtro IIR pasa-bajas para IQ sintético

Este cuaderno crea cinco ejemplos IQ con el formato canónico `(N, 2, L)`, aplica un Butterworth IIR de orden 2 sobre el eje temporal y compara las señales antes y después del filtrado.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal


N, L, SEED = 5, 1_000, 42
CUTOFF = 0.1  # Normalized to the Nyquist frequency.
FILTER_ORDER = 2
OUTPUT_PATH = Path("sebato/filtered_iq.npz")

rng = np.random.default_rng(SEED)
time = np.arange(L, dtype=np.float32)
phase = 2 * np.pi * 0.04 * time
i_signal = np.sin(phase)[None, :] + 0.65 * rng.standard_normal((N, L))
q_signal = np.cos(phase)[None, :] + 0.65 * rng.standard_normal((N, L))
X = np.stack((i_signal, q_signal), axis=1).astype(np.float32)

print(f"Synthetic IQ shape: {X.shape}")
print(f"Synthetic IQ dtype: {X.dtype}")

In [ ]:
def compute_power(iq_array: np.ndarray) -> np.ndarray:
    """Compute mean I² + Q² for each example."""
    i_component = iq_array[:, 0, :]
    q_component = iq_array[:, 1, :]
    return np.mean(i_component**2 + q_component**2, axis=1)


b, a = signal.butter(FILTER_ORDER, CUTOFF, btype="lowpass")
X_filtered = signal.lfilter(b, a, X, axis=2).astype(np.float32)
power_before = compute_power(X)
power_after = compute_power(X_filtered)
power_ratio = power_after.mean() / power_before.mean()

print(f"Butterworth b: {b}")
print(f"Butterworth a: {a}")
print(f"Mean power before: {power_before.mean():.4f}")
print(f"Mean power after:  {power_after.mean():.4f}")
print(f"Power ratio:        {power_ratio:.4f}")

In [ ]:
example_index = 0
trace_samples = 200
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(X[example_index, 0, :trace_samples], label="Original", alpha=0.65)
axes[0, 0].plot(X_filtered[example_index, 0, :trace_samples], label="Filtered", linewidth=2)
axes[0, 0].set(title="I time trace", xlabel="Sample", ylabel="Amplitude")
axes[0, 0].legend()

axes[0, 1].plot(X[example_index, 1, :trace_samples], label="Original", alpha=0.65)
axes[0, 1].plot(X_filtered[example_index, 1, :trace_samples], label="Filtered", linewidth=2)
axes[0, 1].set(title="Q time trace", xlabel="Sample", ylabel="Amplitude")
axes[0, 1].legend()

axes[1, 0].scatter(X[example_index, 0], X[example_index, 1], s=8, alpha=0.25, label="Original")
axes[1, 0].scatter(X_filtered[example_index, 0], X_filtered[example_index, 1], s=8, alpha=0.45, label="Filtered")
axes[1, 0].set(title="I/Q constellation", xlabel="I", ylabel="Q")
axes[1, 0].axis("equal")
axes[1, 0].legend()

axes[1, 1].bar(["Before", "After"], [power_before.mean(), power_after.mean()], color=["tab:gray", "tab:blue"])
axes[1, 1].set(title="Mean IQ power", ylabel="Power")

fig.suptitle("2nd-order Butterworth IIR low-pass filter")
fig.tight_layout()
plt.show()

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez(OUTPUT_PATH, X_filtered=X_filtered, b=b, a=a, seed=SEED)
saved = np.load(OUTPUT_PATH)

assert X_filtered.shape == (5, 2, 1000)
assert X_filtered.dtype == np.float32
assert np.array_equal(saved["X_filtered"], X_filtered)
assert power_after.mean() < power_before.mean()

print(f"Saved filtered signal: {OUTPUT_PATH}")
print(f"Verification shape: {X_filtered.shape}")
print(f"Verification dtype: {X_filtered.dtype}")
print(f"Power before: {power_before.mean():.4f}")
print(f"Power after:  {power_after.mean():.4f}")
print(f"Power ratio:  {power_ratio:.4f}")